In [1]:
import pandas as pd

In [2]:
import os
print(os.getcwd())

C:\Users\ADMIN\Projects\SIH\mplads_project\notebooks


In [3]:
recommended = pd.read_csv("../dataset/Works Recommended.csv")
sanctioned = pd.read_csv("../dataset/Works Sanctioned.csv")
completed = pd.read_csv("../dataset/Works Completed.csv")
expenditure = pd.read_csv("../dataset/Expenditure on Completed and On-going Works as on Date.csv")

In [4]:
recommended = recommended[recommended["Sr. No."] != "Grand Total"]
sanctioned = sanctioned[sanctioned["Sr. No."] != "Grand Total"]
completed = completed[completed["Sr. No."] != "Grand Total"]
expenditure = expenditure[expenditure["Sr. No."] != "Grand Total"]

In [5]:
recommended = recommended.rename(columns={
    "Work category": "work_category",
    "WORK": "work",
    "State": "state",
    "IDA": "ida",
    "Hon'ble Members of Parliament": "mp_name",
    "Constituency": "constituency",
    "Work description": "work_description",
    "Recommended date": "recommended_date",
    "RECOMMENDED AMOUNT   ( ₹ )": "recommended_amount",
    "Sanction Date": "sanction_date",
}).drop(columns=["Sr. No."])

sanctioned = sanctioned.rename(columns={
    "Work category": "work_category",
    "Work": "work",
    "State": "state",
    "IDA": "ida",
    "Hon'ble Members of Parliament": "mp_name",
    "Constituency": "constituency",
    "Work description": "work_description",
    "Recommended date": "recommended_date",
    "Sanction Date": "sanction_date",
    "Sanction Amount ( ₹ )": "sanction_amount",
    "Work Status": "work_status",
}).drop(columns=["Sr. No."])

completed = completed.rename(columns={
    "Work Category": "work_category",
    "Work": "work",
    "State": "state",
    "IDA": "ida",
    "Work Description": "work_description",
    "Hon'ble Members of Parliament": "mp_name",
    "Constituency": "constituency",
    "Completion Date": "completion_date",
    "Amount Disbursed ( ₹ )": "amount_disbursed",
}).drop(columns=["Sr. No.", "Image"])

expenditure = expenditure.rename(columns={
    "State": "state",
    "Work": "work_type",
    "Work ID": "work_id",
    "IDA": "ida",
    "Hon'ble Members of Parliament": "mp_name",
    "Constituency": "constituency",
    "Expenditure Date": "expenditure_date",
    "Vendor Name": "vendor_name",
    "Payment Status": "payment_status",
    "Fund Disbursed Amount ( ₹ )": "fund_disbursed_amount",
}).drop(columns=["Sr. No."])

In [6]:
print(recommended.columns.tolist())
print(sanctioned.columns.tolist())
print(completed.columns.tolist())
print(expenditure.columns.tolist())

['work_category', 'work', 'state', 'ida', 'mp_name', 'constituency', 'work_description', 'recommended_date', 'recommended_amount', 'sanction_date']
['work_category', 'work', 'state', 'ida', 'mp_name', 'constituency', 'work_description', 'recommended_date', 'sanction_date', 'sanction_amount', 'work_status']
['work_category', 'work', 'state', 'ida', 'work_description', 'mp_name', 'constituency', 'completion_date', 'amount_disbursed']
['state', 'work_type', 'work_id', 'ida', 'mp_name', 'constituency', 'expenditure_date', 'vendor_name', 'payment_status', 'fund_disbursed_amount']


In [7]:
def extract_project_id(series):
    pid = series.astype(str).str.extract(
        r"(WS/\s*MP\d+/\d{4}-\d{4}/\d+)",
        expand=False
    )
    return pid.str.replace(r"\s+", "", regex=True)

recommended["project_id"] = extract_project_id(recommended["work"])
sanctioned["project_id"] = extract_project_id(sanctioned["work"])
completed["project_id"] = extract_project_id(completed["work"])
expenditure["project_id"] = (
    expenditure["work_id"].astype(str).str.replace(r"\s+", "", regex=True)
)

In [8]:
for name, df in [
    ("recommended", recommended),
    ("sanctioned", sanctioned),
    ("completed", completed),
    ("expenditure", expenditure),
]:
    print(
        name,
        "non-null", df["project_id"].notna().sum(),
        "unique", df["project_id"].nunique(),
        "sample", df["project_id"].dropna().iloc[0]
    )

recommended non-null 4925 unique 4925 sample WS/MP620/2024-2025/133166
sanctioned non-null 10000 unique 10000 sample WS/MP620/2024-2025/133166
completed non-null 34193 unique 34193 sample WS/MP418/2024-2025/133409
expenditure non-null 18000 unique 12595 sample WS/MP18218/2025-2026/233777


In [9]:
def extract_work_type(series):
    return (
        series.astype(str)
        .str.replace(r"WS/\s*MP\d+/\d{4}-\d{4}/\d+\s*-?\s*", "", regex=True)
        .str.strip()
    )

def extract_district(series):
    return (
        series.astype(str)
        .str.split("(", n=1)
        .str[0]
        .str.strip()
        .str.upper()
    )

for df in [recommended, sanctioned, completed]:
    df["work_type"] = extract_work_type(df["work"])
    df["district"] = extract_district(df["ida"])

expenditure["district"] = extract_district(expenditure["ida"])

In [10]:
print(sanctioned[["work", "project_id", "work_type", "ida", "district"]].head(3))
print("unique work_type sanctioned:", sanctioned["work_type"].nunique())
print(sanctioned["work_type"].value_counts().head(5))
print("unique district sanctioned:", sanctioned["district"].nunique())

                                                work  \
0  WS/\t MP620/2024-2025/133166-Construction of b...   
1  WS/\t MP620/2025-2026/133167-Construction of r...   
2  WS/\t MP620/2024-2025/133190-Construction of b...   

                  project_id  \
0  WS/MP620/2024-2025/133166   
1  WS/MP620/2025-2026/133167   
2  WS/MP620/2024-2025/133190   

                                           work_type  \
0  Construction of buildings for community cultur...   
1  Construction of rooms and halls in school and ...   
2  Construction of buildings for community cultur...   

                                        ida district  
0  DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA)  DHARWAD  
1  DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA)  DHARWAD  
2  DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA)  DHARWAD  
unique work_type sanctioned: 80
work_type
Street lights                                                                                    1989
Construction of roads, link roads, pathways or any other 

In [11]:
date_map = {
    "recommended": recommended,
    "sanctioned": sanctioned,
    "completed": completed,
    "expenditure": expenditure,
}

for df in [recommended, sanctioned]:
    for col in ["recommended_date", "sanction_date"]:
        df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

completed["completion_date"] = pd.to_datetime(
    completed["completion_date"], errors="coerce", dayfirst=True
)
expenditure["expenditure_date"] = pd.to_datetime(
    expenditure["expenditure_date"], errors="coerce", dayfirst=True
)

def to_amount(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )

recommended["recommended_amount"] = to_amount(recommended["recommended_amount"])
sanctioned["sanction_amount"] = to_amount(sanctioned["sanction_amount"])
completed["amount_disbursed"] = to_amount(completed["amount_disbursed"])
expenditure["fund_disbursed_amount"] = to_amount(expenditure["fund_disbursed_amount"])

In [12]:
exp_summary = (
    expenditure.groupby("project_id", as_index=False)
    .agg(
        total_expenditure=("fund_disbursed_amount", "sum"),
        payment_count=("project_id", "count"),
    )
)

print(exp_summary.shape)
print(exp_summary.head())
print(exp_summary["payment_count"].describe())

(12595, 3)
                  project_id  total_expenditure  payment_count
0  WS/MP005/2025-2026/174442             200000              1
1  WS/MP005/2025-2026/175204             250000              1
2  WS/MP005/2025-2026/175542             200000              1
3  WS/MP005/2025-2026/183135             450000              1
4  WS/MP005/2025-2026/194320             299994              1


count    12595.000000
mean         1.429139
std          1.862758
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         35.000000
Name: payment_count, dtype: float64


In [13]:
recommended = recommended[recommended["project_id"].notna()].drop_duplicates("project_id")
sanctioned = sanctioned[sanctioned["project_id"].notna()].drop_duplicates("project_id")
completed = completed[completed["project_id"].notna()].drop_duplicates("project_id")
exp_summary = exp_summary[exp_summary["project_id"].notna()].drop_duplicates("project_id")

In [14]:
print(recommended.shape)
print(sanctioned.shape)
print(completed.shape)
print(exp_summary.shape)

(4925, 13)
(10000, 14)
(34193, 12)
(12595, 3)


In [15]:
id_cols = [
    "state", "district", "constituency", "mp_name", "ida",
    "work_category", "work_type", "work_description",
]

san_keep = ["project_id"] + id_cols + [
    "recommended_date", "sanction_date", "sanction_amount", "work_status"
]
rec_keep = ["project_id", "recommended_amount"]
com_keep = ["project_id", "completion_date", "amount_disbursed"] + id_cols
exp_id_keep = ["project_id", "state", "district", "constituency", "mp_name", "ida", "work_type"]

master = sanctioned[san_keep].merge(
    recommended[rec_keep],
    on="project_id",
    how="outer",
)

master = master.merge(
    completed[com_keep],
    on="project_id",
    how="outer",
    suffixes=("", "_com"),
)

for col in id_cols:
    master[col] = master[col].fillna(master[f"{col}_com"])
    master = master.drop(columns=[f"{col}_com"])

master = master.merge(exp_summary, on="project_id", how="outer")

exp_id = expenditure.drop_duplicates("project_id")[exp_id_keep]
master = master.merge(exp_id, on="project_id", how="left", suffixes=("", "_exp"))

for col in ["state", "district", "constituency", "mp_name", "ida", "work_type"]:
    master[col] = master[col].fillna(master[f"{col}_exp"])
    master = master.drop(columns=[f"{col}_exp"])

In [16]:
print(master.shape)
print(master["project_id"].nunique())
print(master.isna().mean().sort_values(ascending=False).head(10))

(44876, 18)
44876
recommended_amount    0.890253
recommended_date      0.777164
work_status           0.777164
sanction_amount       0.777164
sanction_date         0.777164
payment_count         0.719338
total_expenditure     0.719338
amount_disbursed      0.239972
completion_date       0.238056
work_description      0.176330
dtype: float64


In [17]:
master["cost_amount"] = (
    master["sanction_amount"]
    .fillna(master["recommended_amount"])
    .fillna(master["amount_disbursed"])
    .fillna(master["total_expenditure"])
)

print("cost_amount missing:", master["cost_amount"].isna().sum())
print(master["cost_amount"].describe())

cost_amount missing: 66
count    4.481000e+04
mean     4.881878e+05
std      8.006325e+05
min      3.639000e+03
25%      1.695300e+05
50%      2.994625e+05
75%      5.000000e+05
max      4.647040e+07
Name: cost_amount, dtype: float64


In [18]:
col_order = [
    "project_id", "state", "district", "constituency", "mp_name", "ida",
    "work_category", "work_type", "work_description", "work_status",
    "recommended_date", "sanction_date", "completion_date",
    "recommended_amount", "sanction_amount", "amount_disbursed",
    "total_expenditure", "payment_count", "cost_amount",
]
master = master[col_order]

import os
os.makedirs("../data/cleaned", exist_ok=True)
master.to_csv("../data/cleaned/master_projects.csv", index=False)
print("saved", master.shape)

saved (44876, 19)
